# EXPERIMENT: neighbour price features (2-GPU). NOT part of the report

**This notebook is a separate experiment.** It is not `housing_part1.ipynb` (the report notebook) and its results must not be mixed with it. Its features use the response variable (training prices), so check with the instructor that this is allowed before submitting anything from here.

**The idea (as an appraiser does with comparable sales):** look at the training districts closest to each district and use the prices they "sold for". Unlike the cluster-price experiment, the neighbourhood is centred on each district (its k nearest training districts), not a fixed k-means area.

**Columns** (built by the `NeighbourPrice` class, about 10 columns):

* *Neighbour prices:* mean price of the 5, 10, 20 and 50 nearest training districts (the network picks the useful scale).
* *Income adjustment (optional):* neighbours' price per unit of income × this district's income (what a district with *my* income would cost at *this* location, like adjusting comparables for differences), plus the neighbours' average income and my income minus theirs.
* *Label-free:* distance (km) to the nearest and to the 10 nearest training districts, and the number of other training districts at exactly the same location (53.5% of the districts share their coordinates with another one).

**Leakage:** the price columns of training rows are computed **out-of-fold** (the training rows are split into 5 parts, each gets its neighbours from the other 4), so a district never uses its own price; validation/test rows use all the training rows of that fit. Everything is inside the pipelines, so each CV fold recomputes it from its own training rows. The label-free columns use all other training districts (the district itself excluded).

**Steps:** (1) feature ladder with the NN, same 10 folds: (a) neighbour prices, (b) + income adjustment (the income test), (c) + the 1200 k-means similarity columns; (2) feature relevance: are the hand-made feature blocks still useful next to the neighbour prices?; (3) full NN retuning on the final set; (4) comparison with the current best model (NN, k-means 1200/500, CV R² 0.868) on the same folds, and submissions.

Outputs go to `OUTPUT_DIR = .../outputs_EXPERIMENT_neighbour_price`, submissions are called `submission_EXP_neighbour_price...csv`. Networks are trained two at a time, one per GPU (see `housing_part1_2gpu.ipynb`).

# Packages

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics.pairwise import rbf_kernel
import itertools
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.neighbors import NearestNeighbors
import time
from tqdm.auto import tqdm

# For NN: Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using acceleration device: {device}")

# General settings and data import

In [ ]:
SEED=118

#from google.colab import drive # Only if running from Colab
#drive.mount('/content/drive') # Only if running from Colab
DATA_DIR = "/kaggle/working" # Change to appropriate local path
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs_EXPERIMENT_neighbour_price") # separate from the report outputs
os.makedirs(OUTPUT_DIR, exist_ok=True) # If the folder doesn't exist, create it

x_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_train_houses.csv")
y_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/y_train_houses.csv")
x_test_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_test_houses.csv")

features=x_train_df.columns[1:].tolist()
y_tr_raw=y_train_df.iloc[:, 1].values

# IDA and EDA

In [ ]:
x_train = x_train_df[features]
x_test = x_test_df[features]

# 1. Look for missing data
print(x_train.isna().sum())
print(x_test.isna().sum()) # there is missing data in total_bedrooms

# 2. Numerical variables' distributions
print(x_train.describe())
x_train.assign(Price=y_tr_raw).hist(bins=40, figsize=(12,8))
plt.tight_layout()
plt.show()

# 3. Plot house on a map based on latitude/longitude
cities=np.array([[34.1141,-118.4068],   # Los Angeles
                 [38.5677,-121.4685],   # Sacramento
                 [32.8313,-117.1222],   # San Diego
                 [37.7558,-122.4449],   # San Francisco
                 [37.3012,-121.8480]])  # San Jose
# source: https://simplemaps.com/data/us-cities
city_names = ["Los Angeles", "Sacramento", "San Diego", "San Francisco", "San Jose"]

plt.figure(figsize=(8,8))
plt.scatter(x_train["longitude"], x_train["latitude"], c=y_tr_raw, alpha=0.4, s=7)
plt.colorbar(label="Price")
plt.scatter(cities[:, 1], cities[:, 0], color="red", edgecolor="white", s=150, marker="*")
for i, name in enumerate(city_names):
    txt = plt.text(cities[i, 1] + 0.15, cities[i, 0] - 0.05, name, color="black", fontsize=10, fontweight="bold")
    txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground="white")])
plt.gca().set_aspect("equal")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()
# Expensive houses near the big cities. An engineered feature that measures distance from them could have improving effects?

# 4. Mean price by Ocean proximity
sns.boxplot(data=x_train.assign(Price=y_tr_raw), x="ocean_proximity", y="Price",
            order=x_train.assign(Price=y_tr_raw).groupby("ocean_proximity")["Price"].mean().sort_values(ascending=False).index)
plt.show()
# Expensive houses are located near the coast

# 5. Correlations
heatmap = sns.heatmap(x_train.assign(Price=y_tr_raw).select_dtypes("number").corr(), cmap="coolwarm", annot=True)
## Correlations with response variable "Price"
corr_price = x_train.assign(Price=y_tr_raw).select_dtypes("number").corr()["Price"].sort_values(key=abs, ascending=False)
corr_price
# most correlated variable: median_income

# Feature engineering

In [ ]:
# Histograms: the totals measure district size -> per-household ratios describe the typical home
def ratios(d):
    return d.assign(rooms_per_household=d["total_rooms"]/d["households"],
                    population_per_household=d["population"]/d["households"],
                    bedrooms_per_room=d["total_bedrooms"]/d["total_rooms"],
                    #income_per_room=d["median_income"]/d["total_rooms"]
                   )

def new_features(d):
    return d.assign(income_x_age=d["median_income"]*d["housing_median_age"], # Old houses, high income -> possible wealthy area
                    income_per_room=d["median_income"]/d["rooms_per_household"], # income per unit of housing space
                   )

# Lat/long as a smooth 3D position on the sphere, to account for the earth's curvature
def sphere_coords(d):
    lat=np.radians(d["latitude"])
    lon=np.radians(d["longitude"])
    return d.assign(x_coord=np.cos(lat)*np.cos(lon),
                    y_coord=np.cos(lat)*np.sin(lon),
                    z_coord=np.sin(lat))

# Map: expensive houses are near the big cities -> distance to the 2 closest main cities
def dist_city(d):
    dists = cdist(d[["latitude", "longitude"]].to_numpy(), cities)
    sorted_dists = np.sort(dists, axis=1)
    return d.assign(dist_nearest_city=sorted_dists[:, 0],
                    dist_2nd_nearest_city=sorted_dists[:, 1])

# Map: split CA into k-means clusters, and score each district by its similarity (closeness) to each cluster centroid.
# Clusters use coordinates only (no sample weights), so these features contain no price information.
# KM_GAMMA and KM_CLUSTERS were chosen by cross-validation (see the markdown below); they are the default everywhere.
KM_GAMMA, KM_CLUSTERS = 500, 1200

def kmeans_pipeline(gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    kmeans = KMeans(n_clusters, n_init=10, random_state=SEED)
    return make_pipeline(kmeans, FunctionTransformer(lambda d: np.exp(-gamma * d ** 2)))

# ---------- EXPERIMENT: neighbour prices ----------
class NeighbourPrice(BaseEstimator, TransformerMixin):
    # X columns (in this order): latitude, longitude, median_income.
    # Price columns (use the response): mean price of the k nearest TRAINING districts, for k in ks;
    #   with income=True also the income adjustment: neighbours' price per unit of income x this district's income (k=10, 20),
    #   the neighbours' average income and this district's income minus theirs (k=10).
    #   Training rows (fit_transform): out-of-fold (n_splits parts, each gets its neighbours from the others),
    #   so a district never uses its own price. New rows (transform): neighbours among all the training rows.
    # Label-free columns: distance (km) to the nearest and to the 10 nearest training districts, and the number of other
    #   training districts at exactly the same location (the district itself excluded).
    def __init__(self, ks=(5, 10, 20, 50), income=False, n_splits=5):
        self.ks, self.income, self.n_splits = ks, income, n_splits

    @staticmethod
    def _km(X):
        # (latitude, longitude) in km: 1 degree of latitude ~ 111 km, 1 degree of longitude ~ 111 km x cos(latitude)
        return np.column_stack([X[:, 0] * 111.0, X[:, 1] * 111.0 * np.cos(np.radians(X[:, 0]))])

    def _price_cols(self, X_ref, y_ref, X):
        # price columns of the rows X, with neighbours taken among the reference rows X_ref (prices y_ref)
        _, ind = NearestNeighbors(n_neighbors=max(self.ks)).fit(self._km(X_ref)).kneighbors(self._km(X))
        p = y_ref[ind]
        cols = [p[:, :k].mean(axis=1) for k in self.ks]
        if self.income:
            inc = X_ref[ind, 2]
            cols += [(p / inc)[:, :k].mean(axis=1) * X[:, 2] for k in (10, 20)]
            cols += [inc[:, :10].mean(axis=1), X[:, 2] - inc[:, :10].mean(axis=1)]
        return np.column_stack(cols)

    def _density_cols(self, X, own_rows):
        dist, ind = self.index_.kneighbors(self._km(X), n_neighbors=11)
        if own_rows: # training rows: move the district itself to the end (by position, so same-location twins are kept)
            order = np.argsort(ind == np.arange(len(X))[:, None], axis=1, kind="stable")
            dist = np.take_along_axis(dist, order, axis=1)
        dist = dist[:, :10]
        return np.column_stack([dist[:, 0], dist.mean(axis=1), (dist < 1e-6).sum(axis=1)])

    def fit(self, X, y):
        self.X_, self.y_ = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        self.index_ = NearestNeighbors().fit(self._km(self.X_))
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.column_stack([self._price_cols(self.X_, self.y_, X), self._density_cols(X, own_rows=False)])

    def fit_transform(self, X, y):
        self.fit(X, y)
        X, y = self.X_, self.y_
        prices = None
        for tr, va in KFold(self.n_splits, shuffle=True, random_state=SEED).split(X):
            cols = self._price_cols(X[tr], y[tr], X[va])
            if prices is None:
                prices = np.empty((len(X), cols.shape[1]))
            prices[va] = cols
        return np.column_stack([prices, self._density_cols(X, own_rows=True)])

# Model utilities (same as the report notebook, plus the neighbour-price options)

In [ ]:
cv=KFold(10, shuffle=True, random_state=SEED) # same folds for every model -> paired comparisons
scoring={"mse":"neg_mean_squared_error", "mae":"neg_mean_absolute_error", "r2":"r2"}

# steps: row-wise feature functions, applied first (they only use their own row, so no leakage).
# Everything that is fitted (imputer, scalers, ridge) lives inside the pipeline,
# so cross_validate refits it on each training fold only.
# EXPERIMENT: kmeans = False / True (report behaviour) or a string combining
#   "km" (1200 k-means similarity columns), "nb" (neighbour prices), "inc" (income adjustment), e.g. "nb+inc+km"
def location_parts(kmeans, gamma, n_clusters, keep_income):
    mode = {False: "", True: "km"}.get(kmeans, kmeans)
    parts = []
    if "km" in mode:
        parts.append(("km", kmeans_pipeline(gamma, n_clusters), ["latitude", "longitude"]))
    if "nb" in mode:
        parts.append(("nb", make_pipeline(NeighbourPrice(income="inc" in mode), StandardScaler()),
                      ["latitude", "longitude", "median_income"]))
        if keep_income: # the "nb" step uses median_income, so it must be passed on as a normal feature explicitly
            parts.append(("income", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["median_income"]))
    return parts

def build(steps, model, degree=1, kmeans=False, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    cat_pipeline = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    preprocessing = ColumnTransformer(
        [("cat", cat_pipeline, make_column_selector(dtype_exclude="number"))] +
        location_parts(kmeans, gamma, n_clusters, keep_income=True),
        remainder=num_pipeline)
    return make_pipeline(*[FunctionTransformer(f) for f in steps], preprocessing, model)

def cross_val_all(models, n_jobs=-1):
    return {name:cross_validate(m, x_train, y_tr_raw, cv=cv, scoring=scoring, return_train_score=True, n_jobs=n_jobs)
            for name, m in models.items()}

# train_* = in-sample, cv_* = out-of-sample (mean over the 10 folds).
# rmse_gain: positive = better than the reference row. folds_better: in how many of the 10 folds it beat the reference.
# Reference = the row above, or the first row if vs_first=True (the first row compares with itself).
def summarise(res, vs_first=False):
    names=list(res)
    mse={n:-res[n]["test_mse"] for n in names}
    out=pd.DataFrame(index=names)
    out["train_rmse"]=[np.sqrt(-res[n]["train_mse"].mean()) for n in names]
    out["train_r2"]=[res[n]["train_r2"].mean() for n in names]
    out["cv_rmse"]=[np.sqrt(mse[n].mean()) for n in names]
    out["cv_mae"]=[-res[n]["test_mae"].mean() for n in names]
    out["cv_r2"]=[res[n]["test_r2"].mean() for n in names]
    ref=[names[0] if vs_first else names[max(i-1, 0)] for i in range(len(names))]
    out["rmse_gain"]=[out.loc[r, "cv_rmse"] - out.loc[n, "cv_rmse"] for n, r in zip(names,ref)]
    out["folds_better"]=[int((mse[n] < mse[r]).sum()) for n, r in zip(names, ref)]
    return out.round(3)

In [ ]:
keep_steps=[ratios, new_features, dist_city, sphere_coords]

# Feature sets: the report's best one, and the experimental ones
feats = {"engineered_kmeans": (keep_steps, True),           # report: 1200 k-means similarity columns
         "engineered_nb": (keep_steps, "nb"),               # (a) neighbour prices instead
         "engineered_nb_inc": (keep_steps, "nb+inc"),       # (b) + income adjustment
         "engineered_nb_km": (keep_steps, "nb+km"),         # (c) (a) + the 1200 similarity columns
         "engineered_nb_inc_km": (keep_steps, "nb+inc+km")} # (c) (b) + the 1200 similarity columns

In [ ]:
# Polynomial terms on all original numeric columns; dummies and k-means distances enter linearly.
# Ridge is included otherwise the error explodes at higher degrees.
# Degree 1 is included as a ridge control, so any gain at degree 2-3 is due to the polynomial terms.
def build_poly(steps, model, degree=1, kmeans=True, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    original_numeric = ["housing_median_age", "total_rooms", "total_bedrooms", "population",
                    "households", "median_income", "latitude", "longitude"]

    cat_pipeline = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    original_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                          PolynomialFeatures(degree, include_bias=False), StandardScaler())
    other_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    parts = [("original_num_poly", original_num_pipeline, original_numeric),
             ("cat", cat_pipeline, make_column_selector(dtype_exclude="number"))]
    parts += location_parts(kmeans, gamma, n_clusters, keep_income=False) # median_income is already in original_numeric
    preprocessing = ColumnTransformer(parts, remainder=other_num_pipeline)
    return make_pipeline(*[FunctionTransformer(f) for f in steps], preprocessing, model)

ridge=RidgeCV(alphas=np.logspace(-2, 4, 25))

# Neural network

In [ ]:
# Utility functions
def to_tensor(array, dev=None):
    return torch.tensor(np.asarray(array), dtype=torch.float32, device=dev or device)

def parse_architecture(arch_str):
    # converts a string like "128-64" into a tuple (128, 64)
    if arch_str == "linear":
        return ()
    return tuple(int(w) for w in arch_str.split("-"))

# Preprocessing (imputer, scalers, k-means, one-hot) is fitted on the fitting rows ONLY,
# then applied to every other set (early-stopping, validation and test rows)
def prep_fit(steps, X_fit, y_fit, *other_splits, kmeans=False, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    preprocessor = build(steps, "passthrough", kmeans=kmeans, gamma=gamma, n_clusters=n_clusters)
    X_fit_transformed = preprocessor.fit_transform(X_fit, y_fit)
    other_transformed = [preprocessor.transform(X) for X in other_splits]
    return [X_fit_transformed] + other_transformed

# 1. Construct the network: made it flexible to allow testing for multiple architectures
def build_network(n_inputs, hidden_widths, activation, dropout, use_batchnorm=False):
    # hidden_widths=() gives linear regression
    layers=[]
    in_features=n_inputs

    for width in hidden_widths:
        layers.append(nn.Linear(in_features, width))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(ACTIVATIONS[activation]())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_features=width

    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers)

# 2. Train the network
def train_network(X_train, y_train, X_val, y_val, hidden_widths=(128, 64), activation="relu",
                  learning_rate=1e-3, dropout=0.1, weight_decay=1e-4, batch_size=256, use_batchnorm=False,
                  seed=SEED, max_epochs=300, patience=30, device_name=None):
    # Train with MSE loss and AdamW. LR halves on plateau; stops early on validation RMSE and restores the best epoch's weights
    torch.manual_seed(seed)
    dev = torch.device(device_name) if device_name else device # [2-GPU] the GPU this network trains on

    # standardize the target using training statistics only
    y_mean, y_std = y_train.mean(), y_train.std()
    X_train_t = to_tensor(X_train, dev)
    X_val_t = to_tensor(X_val, dev)
    y_train_t = to_tensor((y_train-y_mean)/y_std, dev).unsqueeze(1)

    net = build_network(X_train_t.shape[1], hidden_widths, activation, dropout, use_batchnorm).to(dev)
    optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
    loss_fn = nn.MSELoss()

    def compute_rmse(X, y_true):
        # RMSE in dollars, with dropout/batchnorm turned off
        net.eval()
        with torch.no_grad():
            preds = net(X).squeeze(1).cpu().numpy() * y_std + y_mean
        return np.sqrt(np.mean((preds - y_true) ** 2))

    best_val_rmse = np.inf
    best_epoch = 0
    best_weights = None
    history = []

    for epoch in range(max_epochs):
        net.train()
        shuffled_idx = torch.randperm(len(X_train_t), device=dev)

        for start in range(0, len(shuffled_idx), batch_size):
            batch_idx = shuffled_idx[start:start + batch_size]
            if len(batch_idx) < 2:
                continue  # batchnorm needs at least 2 rows
            optimizer.zero_grad()
            loss = loss_fn(net(X_train_t[batch_idx]), y_train_t[batch_idx])
            loss.backward()
            optimizer.step()

        train_rmse = compute_rmse(X_train_t, y_train)
        val_rmse = compute_rmse(X_val_t, y_val)
        history.append((train_rmse, val_rmse))
        scheduler.step(val_rmse)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_weights = {k: v.clone() for k, v in net.state_dict().items()}
        elif epoch - best_epoch >= patience:
            break

    net.load_state_dict(best_weights)
    return {"net": net, "y_mean": y_mean, "y_std": y_std, "best_epoch": best_epoch, "history": history}

# 3. Use the network to make predictions
def predict(result, X):
    result["net"].eval()
    dev = next(result["net"].parameters()).device # [2-GPU] predict on the network's own GPU
    with torch.no_grad():
        preds = result["net"](to_tensor(X, dev)).squeeze(1).cpu().numpy()
    return preds * result["y_std"] + result["y_mean"]

# 4. Tuning
def run_grid(configs, feature_set_name):
    # [2-GPU] same networks as the 1-GPU version, trained in parallel by train_many (one worker per GPU)
    X_train, X_val = data[feature_set_name]
    n_train = len(ya)
    arrays = {"X": np.vstack([X_train, X_val]).astype(np.float32), "y": np.concatenate([ya, yb])}
    fit, stop = np.arange(n_train), np.arange(n_train, n_train + len(yb))
    jobs = [dict(X="X", y="y", fit=fit, stop=stop, predict=[("X", fit), ("X", stop)], cfg=config) for config in configs]
    results_rows = []
    histories = []

    for config, res in zip(configs, train_many(arrays, jobs)):
        pred_train, pred_val = res["preds"]
        arch_label = "-".join(map(str, config["hidden_widths"])) or "linear"
        other_params = {k: v for k, v in config.items() if k != "hidden_widths"}
        results_rows.append({
            "feat": feature_set_name,
            "arch": arch_label,
            **other_params,
            "best_epoch": res["best_epoch"] + 1,
            "train_rmse": np.sqrt(mean_squared_error(ya, pred_train)),
            "val_rmse": np.sqrt(mean_squared_error(yb, pred_val)),
            "val_mae": mean_absolute_error(yb, pred_val),
            "val_r2": r2_score(yb, pred_val),
            "secs": res["secs"],
        })
        histories.append(res["history"])
    return pd.DataFrame(results_rows), histories

# For visualization purposes:
def show_grid(df, rows, cols, value="val_rmse"):
    # Pivot into a rows x cols table of validation RMSE, colored (darker=lower error)
    return df.pivot_table(index=rows, columns=cols, values=value).round(0).style.background_gradient(cmap="viridis_r", axis=None)

def plot_curves(histories, titles):
    # Diagnostic plot for convergence
    n_plots = len(histories)
    n_cols = 4
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    lowest_val_rmse = min(min(val for _, val in hist) for hist in histories)

    for ax, hist, title in zip(axes, histories, titles):
        hist = np.array(hist)
        ax.plot(hist[:, 0], label="train")
        ax.plot(hist[:, 1], label="validation")
        ax.set_title(title)
        ax.set_ylim(lowest_val_rmse * 0.6, lowest_val_rmse * 2)

    # hide unused subplot slots
    for ax in axes[n_plots:]:
        ax.axis("off")

    axes[0].legend()
    fig.supxlabel("epoch")
    fig.supylabel("RMSE ($)")
    plt.tight_layout()
    plt.show()

In [ ]:
# [2-GPU] Train several networks at the same time: one worker PROCESS per GPU.
# Only the order of the work changes: every network gets exactly the same rows, configuration and seed as in
# housing_part1.ipynb, so the results are the same (up to the usual GPU rounding differences).
# Processes, not threads: each process has its own random-number generators, so a network's seed controls only that network.
# A job = one network: row indices (fit / stop) into a matrix of `arrays`, the configuration (incl. seed),
# and the matrices (or rows) to predict. Large arrays are shared with the workers through files (joblib memmapping).
from joblib import Parallel, delayed

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("networks are trained on:", DEVICES)

def _train_chunk(arrays, jobs, device_name):
    # runs inside one worker: trains its jobs one after the other on one device
    out = []
    for job in jobs:
        start_time = time.time()
        X, y = arrays[job["X"]], arrays[job["y"]]
        result = train_network(X[job["fit"]], y[job["fit"]], X[job["stop"]], y[job["stop"]], device_name=device_name, **job["cfg"])
        preds = [predict(result, arrays[name] if rows is None else arrays[name][rows]) for name, rows in job["predict"]]
        out.append(dict(preds=preds, best_epoch=result["best_epoch"], history=result["history"], secs=time.time() - start_time))
    return out

def train_many(arrays, jobs):
    # splits the jobs over the devices (job i -> device i % n_devices) and returns one result per job, in the original order
    chunks = [list(range(d, len(jobs), len(DEVICES))) for d in range(len(DEVICES))]
    if len(DEVICES) == 1:
        parts = [_train_chunk(arrays, jobs, DEVICES[0])]
    else:
        parts = Parallel(n_jobs=len(DEVICES))(delayed(_train_chunk)(arrays, [jobs[i] for i in chunk], dev)
                                              for chunk, dev in zip(chunks, DEVICES))
    results = [None] * len(jobs)
    for chunk, part in zip(chunks, parts):
        for i, res in zip(chunk, part):
            results[i] = res
    return results

In [ ]:
# Tuning functions
def to_cfg(row):
    # one row / dict of SPACE values -> keyword arguments for train_network
    return dict(hidden_widths=parse_architecture(row["arch"]), activation=row["activation"],
                learning_rate=float(row["learning_rate"]), dropout=float(row["dropout"]),
                weight_decay=float(row["weight_decay"]), batch_size=int(row["batch_size"]),
                use_batchnorm=bool(row["use_batchnorm"]))

def sample_configs(space, n, seed):
    # same random configurations every time (seeded), so a resumed run continues the same list
    rng = np.random.default_rng(seed)
    seen = set()
    configs = []
    while len(configs) < n:
        c = {k: v[rng.integers(len(v))] for k, v in space.items()}
        key = tuple(c.values())
        if key not in seen:
            seen.add(key)
            configs.append(c)
    return configs

def cached_grid(filename, configs, feat, every=10):
    # run_grid with a CSV checkpoint every `every` networks; resumes after a Colab disconnect
    path = os.path.join(OUTPUT_DIR, filename)
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    for i in range(len(done), len(configs), every):
        part, _ = run_grid(configs[i:i + every], feat)
        done = pd.concat([done, part], ignore_index=True)
        done.to_csv(path, index=False)
    return done

def screen_survivors(screen, space, keep):
    # best `keep[dim]` values of each hyperparameter, ranked by mean validation RMSE
    survivors = {}
    for dim in space:
        stats = screen.groupby(dim)["val_rmse"].agg(mean="mean", median="median", best="min", n="count")
        stats = stats.sort_values("median")
        display(dim, stats.round(0))
        survivors[dim] = list(stats.index[:keep[dim]])
    return survivors

def recheck_top(final, feat, n_recheck, space, seeds):
    # re-run the best `n_recheck` networks with new seeds, since a single run can be noisy
    rows = []
    for _, row in final.sort_values("val_rmse").head(n_recheck).iterrows():
        repeats, _ = run_grid([{**to_cfg(row), "seed": s} for s in seeds], feat)
        rows.append(dict(
            **{k: row[k] for k in space},
            step2_mean=row["val_rmse"],
            mean_seeds=repeats["val_rmse"].mean(),
            sd_seeds=repeats["val_rmse"].std(),
            mean_best_epoch=repeats["best_epoch"].mean(),
        ))
    return pd.DataFrame(rows).sort_values("mean_seeds")

def tune_nn(feat, space, n_screen, keep, max_final, n_recheck, recheck_seeds, seed, tag, step2_seeds):
    # step 1: broad random screen over the whole space
    screen = cached_grid(f"nn_screen_{tag}.csv", [to_cfg(c) for c in sample_configs(space, n_screen, seed)], feat)
    survivors = screen_survivors(screen, space, keep)
    print("surviving values:", survivors)

    # step 2: full grid over the survivors only (subsampled if larger than max_final)
    combos = list(itertools.product(*survivors.values()))
    if len(combos) > max_final:
        pick = np.random.default_rng(seed).choice(len(combos), max_final, replace=False)
        combos = [combos[i] for i in np.sort(pick)]

    configs=[to_cfg(dict(zip(survivors, c))) for c in combos]
    runs=[cached_grid(f"nn_final_{tag}_seed{s}.csv", [{**c, "seed":s} for c in configs], feat) for s in step2_seeds]
    # one row per configuration, every result column averaged over the step-2 seeds
    metric_cols=["best_epoch", "train_rmse", "val_rmse", "val_mae", "val_r2", "secs"]
    final=pd.concat(runs).groupby(list(space), as_index=False, sort=False)[metric_cols].mean()

    display(final.sort_values("val_rmse").head(15).round(3))
    display(show_grid(final, "arch", ["activation", "learning_rate"]))
    display(show_grid(final, "dropout", ["weight_decay", "batch_size"]))

    # step 3: re-run the best few with new seeds, since a single run is noisy
    recheck = recheck_top(final, feat, n_recheck, space, recheck_seeds)
    display(recheck.round(3))
    return to_cfg(recheck.iloc[0]), screen, final, recheck

In [ ]:
def cross_validate_nn(cfg, feat, seeds=(89, 233, 1597)):
    # Honest CV: each fold trains a fresh ensemble of `seeds` networks, with preprocessing fit on that
    # fold's training rows only. Early stopping watches a held-out 10% split (`stop`), never the
    # validation fold (`val`), so nothing about the validation fold leaks into training decisions.
    # Keys match cross_validate's naming (train_mse/test_mse/...) so summarise() works on both.
    # [2-GPU] the preprocessing of all folds is done first, then all 10 x 3 networks are trained in parallel.
    steps, km = feats[feat]
    y = y_tr_raw
    fold_scores = dict(train_mse=[], test_mse=[], test_mae=[], train_r2=[], test_r2=[])
    test_preds = []
    oof_preds = np.zeros(len(y))

    arrays, jobs, folds = {}, [], []
    for f, (train_idx, val_idx) in enumerate(tqdm(list(cv.split(x_train)), desc="preprocessing folds")):
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=SEED)
        X_fit, X_stop, X_val, X_test = prep_fit(
            steps, x_train.iloc[fit_idx], y[fit_idx], x_train.iloc[stop_idx],
            x_train.iloc[val_idx], x_test, kmeans=km)
        # rows of fold f stacked as [fit | stop | val]; the test set in its own matrix
        arrays[f"X{f}"] = np.vstack([X_fit, X_stop, X_val]).astype(np.float32)
        arrays[f"y{f}"] = np.concatenate([y[fit_idx], y[stop_idx], y[val_idx]])
        arrays[f"T{f}"] = np.asarray(X_test, dtype=np.float32)
        n_fit, n_stop = len(fit_idx), len(stop_idx)
        r_fit, r_stop, r_val = np.arange(n_fit), np.arange(n_fit, n_fit + n_stop), np.arange(n_fit + n_stop, n_fit + n_stop + len(val_idx))
        for seed in seeds:
            jobs.append(dict(X=f"X{f}", y=f"y{f}", fit=r_fit, stop=r_stop,
                             predict=[(f"X{f}", r_fit), (f"X{f}", r_val), (f"T{f}", None)], cfg={**cfg, "seed": seed}))
        folds.append((fit_idx, val_idx))

    results = train_many(arrays, jobs)

    for f, (fit_idx, val_idx) in enumerate(folds):
        fold_results = results[f * len(seeds):(f + 1) * len(seeds)]
        fit_pred = np.mean([r["preds"][0] for r in fold_results], axis=0)  # ensemble = average of the seeds
        val_pred = np.mean([r["preds"][1] for r in fold_results], axis=0)
        test_preds.append(np.mean([r["preds"][2] for r in fold_results], axis=0))
        oof_preds[val_idx] = val_pred

        fold_scores["train_mse"].append(-mean_squared_error(y[fit_idx], fit_pred))
        fold_scores["test_mse"].append(-mean_squared_error(y[val_idx], val_pred))
        fold_scores["test_mae"].append(-mean_absolute_error(y[val_idx], val_pred))
        fold_scores["train_r2"].append(r2_score(y[fit_idx], fit_pred))
        fold_scores["test_r2"].append(r2_score(y[val_idx], val_pred))

    fold_scores = {k: np.array(v) for k, v in fold_scores.items()}
    return fold_scores, np.mean(test_preds, axis=0), oof_preds

In [ ]:
# Features to test
ACTIVATIONS = {"relu":nn.ReLU, "leaky_relu":nn.LeakyReLU, "elu":nn.ELU, "silu":nn.SiLU, "gelu":nn.GELU, "tanh":nn.Tanh}
FINAL_FEATURE_SET = "engineered_nb" # EXPERIMENT: set automatically in step 1

# Tuning configuration
"""
Step 1: random search over everything at once; for each value of each hyperparameter we look at the average
  validation RMSE of all the networks that used it. Therefore, a value that is consistently worse is dropped.
Step 2: full search over the "surviving" values (if < MAX_FINAL, otherwise another random search).
Step 3: the best networks are re-run with new seeds and the best average is chosen as the final configuration.
Both steps are saved to CSV in OUTPUT_DIR and resume after a Colab disconnect; delete the CSV files to repeat a step.
"""

# EXPERIMENT configurations:
# REF_CFG   = the report's final NN configuration (for the 1200 k-means similarity columns): the model to beat
# START_CFG = configuration used in the step-1 ladder (tuned with k=75, ~96 inputs, closer to the ~30 inputs of the
#             neighbour-price sets than REF_CFG). The full retuning in step 2 replaces it.
REF_CFG = dict(hidden_widths=(256,128,64), activation="leaky_relu", learning_rate=0.0005, dropout=0.2,
               weight_decay=0.0, batch_size=256, use_batchnorm=True)
START_CFG = dict(hidden_widths=(256,128), activation="relu", learning_rate=0.005, dropout=0.2,
                 weight_decay=0.001, batch_size=128, use_batchnorm=True)
FIXED_CFG = START_CFG # used only if RUN_TUNING = False
RUN_TUNING = True # False: skip the search and use FIXED_CFG
N_SCREEN = 180 # Networks to test in step 1
KEEP = dict(arch=3, activation=3, learning_rate=2, dropout=2,
            weight_decay=1, batch_size=1, use_batchnorm=1) # How many values of each hyperparameter survive step 1
MAX_FINAL = 120 # Maximum number of networks to test in step 2
N_RECHECK = 5 # Top networks from step 2, to test with new seeds at the end to eliminate noise
RECHECK_SEEDS = (9, 28, 630, 2026, 40121, 271828, 7778777)
STEP2_SEEDS=(31337, 65537, 104729)

# Hyperparameter candidates
SPACE = dict(
    arch=["256", "64-64", "128-64", "256-128", "128-64-32", "256-128-64"],
    activation=["relu", "leaky_relu", "elu", "silu", "gelu", "tanh"],
    learning_rate=[1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    dropout=[0, 0.05, 0.1, 0.2, 0.3, 0.4],
    weight_decay=[0, 1e-3, 1e-2, 1e-1, 1],
    batch_size=[64, 128, 256, 512],
    use_batchnorm=[False, True],
)

# Step 1: feature ladder (NN, `START_CFG`, same 10 folds)

* (a) neighbour prices (replacing the 1200 k-means similarity columns)
* (b) (a) + income adjustment: **the income test**. The income block is kept only if it is *clearly* better: lower CV RMSE **and** better in at least 7 of the 10 folds.
* (c) the better of (a)/(b) + the 1200 k-means similarity columns (same rule to keep them).

The report's final NN (`REF_CFG`, k-means 1200/500) is cross-validated on the same folds as the reference. Per-fold results are saved to `OUTPUT_DIR/neighbour_price_tests/`; a re-run loads them instead of retraining.

In [ ]:
NB_TEST_DIR = os.path.join(OUTPUT_DIR, "neighbour_price_tests")
os.makedirs(NB_TEST_DIR, exist_ok=True)
METRICS = ["train_mse", "test_mse", "test_mae", "train_r2", "test_r2"]

def cached_cv(name, cv_fn):
    # cv_fn() -> per-fold scores (cross_validate format); saved to CSV, so a re-run loads them instead of retraining
    path = os.path.join(NB_TEST_DIR, f"{name}.csv")
    if os.path.exists(path):
        d = pd.read_csv(path)
        return {m: d[m].to_numpy() for m in METRICS}
    start_time = time.time()
    scores = cv_fn()
    pd.DataFrame({m: scores[m] for m in METRICS}).to_csv(path, index_label="fold")
    print(f"{name}: cv_rmse={np.sqrt(-np.mean(scores['test_mse'])):,.0f} ({time.time() - start_time:.0f}s)")
    return scores

def clearly_better(new, old):
    # lower CV RMSE and better in at least 7 of the 10 folds
    return (-new["test_mse"].mean() < -old["test_mse"].mean()) and ((new["test_mse"] > old["test_mse"]).sum() >= 7)

In [ ]:
ref_cv = cached_cv("ref_NN_engineered_kmeans", lambda: cross_validate_nn(REF_CFG, "engineered_kmeans")[0])
nb_a = cached_cv("a_engineered_nb", lambda: cross_validate_nn(START_CFG, "engineered_nb")[0])
nb_b = cached_cv("b_engineered_nb_inc", lambda: cross_validate_nn(START_CFG, "engineered_nb_inc")[0])
display(summarise({"(a) neighbour prices": nb_a, "(b) + income adjustment": nb_b})) # (b) vs (a) = the income test

USE_INCOME = None # EDIT: True / False to override the automatic choice
use_income = clearly_better(nb_b, nb_a) if USE_INCOME is None else USE_INCOME
base_set, base_cv = ("engineered_nb_inc", nb_b) if use_income else ("engineered_nb", nb_a)
print("income adjustment kept:", use_income)

nb_c = cached_cv(f"c_{base_set}_km", lambda: cross_validate_nn(START_CFG, base_set + "_km")[0])
display(summarise({f"{base_set}": base_cv, f"{base_set} + k-means similarities": nb_c})) # (c) vs the better of (a)/(b)

CHOSEN_FEATURE_SET = None # EDIT: a key of feats to override the automatic choice
FINAL_FEATURE_SET = CHOSEN_FEATURE_SET or (base_set + "_km" if clearly_better(nb_c, base_cv) else base_set)
step1_cv = {"engineered_nb": nb_a, "engineered_nb_inc": nb_b, base_set + "_km": nb_c} # results by feature set, reused in step 2
print("feature set for the rest of the notebook:", FINAL_FEATURE_SET)

display(summarise({"NN | engineered_kmeans (report, REF_CFG)": ref_cv, "(a) engineered_nb": nb_a,
                   "(b) engineered_nb_inc": nb_b, f"(c) {base_set}_km": nb_c}, vs_first=True))

# Step 2: feature relevance

The neighbour prices measure "how expensive is this location" directly, so some hand-made features (built as proxies for location) may no longer be needed. **Leave-one-block-out** on the step-1 feature set (NN, `START_CFG`, same 10 folds): each block is removed in turn:

* `dist_city` (distance to the 2 nearest big cities)
* `sphere_coords` (3D coordinates)
* `new_features` (income × age, income per room)
* `ratios` (per-household ratios; `new_features` goes with it, because it uses one of the ratio columns)

**Rule (simplicity first):** a block is dropped unless the model without it is *clearly worse* (higher CV RMSE **and** worse in at least 7 of the 10 folds). The raw columns always stay. Then one **confirmation run** with all the dropped blocks removed together: if that is clearly worse than the full set (blocks can overlap), the full set is kept.

In [ ]:
BLOCKS = {"dist_city": [dist_city], "sphere_coords": [sphere_coords], "new_features": [new_features],
          "ratios": [ratios, new_features]} # removing the ratios also removes new_features (it uses rooms_per_household)
full_steps, full_mode = feats[FINAL_FEATURE_SET]
full_cv = step1_cv.get(FINAL_FEATURE_SET) or cached_cv(f"full_{FINAL_FEATURE_SET}", lambda: cross_validate_nn(START_CFG, FINAL_FEATURE_SET)[0])

def without(removed):
    # registers the feature set "FINAL_FEATURE_SET minus the removed blocks" in feats and returns its name
    removed = sorted(removed)
    name = f"{FINAL_FEATURE_SET}_without_" + "_".join(removed)
    drop = [f for b in removed for f in BLOCKS[b]]
    feats[name] = ([f for f in full_steps if f not in drop], full_mode)
    return name

ablation = {}
for block in BLOCKS:
    name = without([block])
    ablation[block] = cached_cv(name, lambda: cross_validate_nn(START_CFG, name)[0])
display(summarise({"all blocks": full_cv, **{f"without {b}": s for b, s in ablation.items()}}, vs_first=True))

DROP_BLOCKS = None # EDIT: a list of block names to override the automatic choice ([] = keep everything)
dropped = DROP_BLOCKS if DROP_BLOCKS is not None else [b for b, s in ablation.items() if not clearly_better(full_cv, s)]
print("blocks that are not clearly useful:", dropped)

if dropped:
    reduced = without(dropped)
    reduced_cv = cached_cv(reduced, lambda: cross_validate_nn(START_CFG, reduced)[0])
    display(summarise({"all blocks": full_cv, f"without {', '.join(dropped)}": reduced_cv}, vs_first=True))
    if DROP_BLOCKS is not None or not clearly_better(full_cv, reduced_cv):
        FINAL_FEATURE_SET = reduced
print("feature set for the rest of the notebook:", FINAL_FEATURE_SET)

# Step 3: NN retuning on the final feature set

Same procedure as the report (random screen → full grid over the surviving values → recheck of the best configurations with new seeds), on an 80/20 split. The tuning CSVs are tagged with the feature set, so they never mix with other runs.

In [ ]:
# Divide into train/validation split for tuning (80/20, random)
idx_tr, idx_va = train_test_split(np.arange(len(y_tr_raw)), test_size=0.2, random_state=SEED)
xa, ya = x_train.iloc[idx_tr], y_tr_raw[idx_tr]
xb, yb = x_train.iloc[idx_va], y_tr_raw[idx_va]

steps, km = feats[FINAL_FEATURE_SET]
data = {FINAL_FEATURE_SET: tuple(prep_fit(steps, xa, ya, xb, kmeans=km))} # EXPERIMENT: only the chosen feature set
print({name: d[0].shape[1] for name, d in data.items()}) # number of input columns per feature set

# reference: the linear model (best features) on this exact split
Xa, Xb = data[FINAL_FEATURE_SET]
lm = LinearRegression().fit(Xa, ya)
print("linear model on this split | train RMSE:", round(np.sqrt(mean_squared_error(ya, lm.predict(Xa)))),
      "| validation RMSE:", round(np.sqrt(mean_squared_error(yb, lm.predict(Xb)))))


In [ ]:
# Tuning
if RUN_TUNING:
    best_cfg, screen, final, recheck = tune_nn(
        feat=FINAL_FEATURE_SET, space=SPACE, n_screen=N_SCREEN, keep=KEEP, max_final=MAX_FINAL,
        n_recheck=N_RECHECK, recheck_seeds=RECHECK_SEEDS, seed=SEED, tag=f"EXP_{FINAL_FEATURE_SET}", step2_seeds=STEP2_SEEDS)
else:
    best_cfg = FIXED_CFG

print(best_cfg)

# Step 4: comparison with the current best model, and submissions

Same 10 folds, every row compared with the first one (the report's final NN: k-means 1200/500 similarity columns, `REF_CFG`).

In [ ]:
exp_cv, exp_test, exp_oof = cross_validate_nn(best_cfg, FINAL_FEATURE_SET)
ridge_exp = cross_val_all({f"ridge deg1 | {FINAL_FEATURE_SET}": build_poly(feats[FINAL_FEATURE_SET][0], ridge, 1, kmeans=feats[FINAL_FEATURE_SET][1])})
display(summarise({"NN | engineered_kmeans (report, current best)": ref_cv, **ridge_exp,
                   f"NN | {FINAL_FEATURE_SET} (experiment, retuned)": exp_cv}, vs_first=True))

def save_submission(test_pred, filename):
    print(filename, "| non-positive predictions before clipping:", (test_pred <= 0).sum(),
          "| above the training max:", (test_pred > y_tr_raw.max()).sum())
    pred = np.clip(test_pred, y_tr_raw.min(), y_tr_raw.max())
    pd.DataFrame({"ID": x_test_df.iloc[:, 0], "Price": pred}).to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

# test predictions of the 30 CV networks above (10 folds x 3 seeds)
save_submission(exp_test, "submission_EXP_neighbour_price_cv.csv")

In [ ]:
# Final NN: preprocessing fitted on ALL training rows. 10 groups x 3 seeds = 30 networks averaged.
# Each group learns from 90% of the rows and early-stops on the other 10%; the 10% rotates (like the CV folds),
# so every row is used for learning and every network still stops at its best epoch on rows it did not learn from.
# [2-GPU] the 30 networks are trained in parallel.
steps, km = feats[FINAL_FEATURE_SET]
X_all, X_test_all = prep_fit(steps, x_train, y_tr_raw, x_test, kmeans=km)

arrays = {"X": np.asarray(X_all, dtype=np.float32), "y": y_tr_raw, "T": np.asarray(X_test_all, dtype=np.float32)}
jobs = [dict(X="X", y="y", fit=fit_idx, stop=stop_idx, predict=[("T", None)], cfg={**best_cfg, "seed": seed})
        for fit_idx, stop_idx in KFold(10, shuffle=True, random_state=SEED).split(X_all)
        for seed in (89, 233, 1597)]
test_preds = [res["preds"][0] for res in train_many(arrays, jobs)]

save_submission(np.mean(test_preds, axis=0), "submission_EXP_neighbour_price.csv")